# Headline figures: benchmark v2 leaderboard

Six models, two runs each, wrote a bidding policy for a 1 MW / 2 MWh battery in the Australian electricity market (NEM). Each run had two hours of active model time and up to 10 submissions, with feedback on 12 seven-day holdout episodes. The leaderboard score is the final policy on the full holdout: 12 months (one of each calendar month, Jan 2023 – Jun 2026) in all five regions, as a percentage of what a perfect-foresight operator earns, counting wear (\\$30/MWh), the value of energy left in storage, and half the daily shortfall against the TWAP baseline.

The figures are drawn with Altair, are interactive here (hover a point for its details) and are saved as PNGs to `docs/figures/` for the README: the default choice below under plain names such as `leaderboard.png`, other choices with a suffix such as `leaderboard_episodes_dollars.png`. The data come from the exported results in `results/v2/`, read by `headline_data.py`.

**Choose what the figures show** in the next cell:
- `SCORE_SET`: `"full"` for the 12-month holdout (the leaderboard score) or `"episodes"` for the 12 seven-day feedback episodes the agents saw.
- `UNITS`: `"pct"` for % of perfect foresight or `"dollars"` for \\$k per MW per year (the score's dollar value, averaged over the five regions' 1 MW batteries). Dollar figures assume a lone price-taking battery and overstate what a fleet could earn; see `price_elasticity.ipynb`.

The LP baseline is left out: it bids setpoints rather than price curves, which says more about bidding than about forecasting.

In [ ]:
SCORE_SET = "full"   # "full" (12-month holdout, the leaderboard score) or "episodes" (12 feedback episodes)
UNITS = "pct"        # "pct" (% of perfect foresight) or "dollars" ($k per MW per year)

In [ ]:
from pathlib import Path
import sys

import altair as alt
import numpy as np
import pandas as pd
import vl_convert as vlc

NB = Path.cwd() if Path.cwd().name == "notebooks" else Path.cwd() / "notebooks"
sys.path.insert(0, str(NB))
from headline_data import SET_LABEL, SHORT, best_so_far, load, pareto

lb = load(SCORE_SET, UNITS)
runs, board = lb.runs, lb.board
TWAP = lb.twap[SCORE_SET]
OUT = NB.parent / "docs" / "figures"
OUT.mkdir(parents=True, exist_ok=True)
SUFFIX = "" if (SCORE_SET, UNITS) == ("full", "pct") else f"_{SCORE_SET}_{UNITS}"

# style: the dataviz reference palette, light mode, one accent hue
FONT = "Noto Sans"
SURFACE, INK, INK2, MUTED, GRID = "#fcfcfb", "#0b0b0b", "#52514e", "#8c8b86", "#ecebe7"
BLUE, BLUE_LIGHT, RED = "#2a78d6", "#d6e5f7", "#e34948"
SOURCE = "Storage Optimization Benchmark v2 · 6 models × 2 runs · 1 MW / 2 MWh battery in the NEM · October 2026"
STEP = 5 if UNITS == "pct" else 10
VEGA_LABEL = "datum.value + '%'" if UNITS == "pct" else "'$' + datum.value + 'k'"


def limits(values, step=STEP, pad=0.0, floor=None):
    v = np.asarray([x for x in values if pd.notna(x)])
    lo = np.floor(v.min() / step - pad) * step if floor is None else floor
    return float(lo), float(np.ceil(v.max() / step + pad) * step)


def run_tooltip(df):
    return df.assign(score_txt=df.score.map(lambda v: lb.fmt(v) if pd.notna(v) else "no submission"),
                     cost_txt=df.cost.map(lambda v: f"${v:.2f}"), run=df.model + " run " + df.rep.astype(str))


# ---- Altair helpers
def a_finish(chart, title, subtitle, width):
    footer = alt.Chart(pd.DataFrame({"t": [SOURCE]})).mark_text(align="left", baseline="top", fontSize=11, color=MUTED, font=FONT) \
        .encode(x=alt.value(0), y=alt.value(0), text="t:N").properties(width=width, height=12)
    return (alt.vconcat(chart, footer, spacing=18)
            .properties(title=alt.TitleParams(title, subtitle=subtitle, anchor="start", fontSize=21, fontWeight=700,
                                              color=INK, subtitleFontSize=13.5, subtitleColor=INK2, offset=16))
            .configure(font=FONT, background=SURFACE, padding={"left": 16, "right": 28, "top": 16, "bottom": 14})
            .configure_view(stroke=None)
            .configure_axis(labelColor=INK2, titleColor=INK2, gridColor=GRID, domainColor=MUTED, tickColor=MUTED,
                            labelFontSize=12, titleFontSize=12.5, titleFontWeight=500, labelPadding=6, titlePadding=12)
            .configure_header(labelFontSize=13.5, labelFontWeight=700, labelColor=INK, labelAnchor="start", title=None))


def a_labels(df, x, y, xscale=None, yscale=None):
    # one text layer per placement; df has label, dx, dy (dy positive = up)
    layers = []
    for (dx, dy), g in df.groupby(["dx", "dy"]):
        align = "center" if dx == 0 else ("left" if dx > 0 else "right")
        baseline = "middle" if dx != 0 else ("bottom" if dy > 0 else "top")
        layers.append(alt.Chart(g).mark_text(align=align, baseline=baseline, dx=dx, dy=-dy, fontSize=12, color=INK2)
                      .encode(x=alt.X(f"{x}:Q", scale=xscale), y=alt.Y(f"{y}:Q", scale=yscale), text="label:N"))
    return alt.layer(*layers)


def a_save(chart, name):
    (OUT / f"{name}{SUFFIX}.png").write_bytes(vlc.vegalite_to_png(chart.to_json(), scale=2))
    return chart

## 1. Leaderboard

In [ ]:
words = {0: "No", 1: "One", 2: "Two", 3: "Three", 4: "Four", 5: "Five", 6: "Six"}
beat = int((board["mean"] > TWAP).sum())
LB_TITLE = f"{words[beat]} model{'s' if beat != 1 else ''} beat the TWAP baseline"
LB_SUB = [f"Final policy on the {SET_LABEL[SCORE_SET]}, {lb.unit_label}.",
          "Dots: each run. Bar: mean of runs that submitted. ×: run with no submission."]
x0, x1 = limits(list(runs.score) + [TWAP], pad=1)
order = board.model.tolist()
bd = board.assign(start=x0, label=board.model + " (" + board.effort + ")", mean_txt=board["mean"].map(lb.fmt))
rd = run_tooltip(runs).merge(bd[["model", "label"]], on="model")
none = rd[rd.score.isna()].assign(x=x0 + (x1 - x0) * 0.02)
W, H = 640, 300
yenc = alt.Y("label:N", sort=bd.label.tolist(), title=None, axis=alt.Axis(domain=False, ticks=False, labelFontSize=13, labelColor=INK, labelPadding=10))
xscale = alt.Scale(domain=[x0, x1], zero=False, nice=False)
bars = alt.Chart(bd).mark_bar(color=BLUE_LIGHT, cornerRadiusEnd=8, height=24).encode(
    x=alt.X("start:Q", scale=xscale, title=f"final policy on the {SET_LABEL[SCORE_SET]}", axis=alt.Axis(labelExpr=VEGA_LABEL, tickCount=8)),
    x2="mean:Q", y=yenc, tooltip=[alt.Tooltip("label:N", title="model"), alt.Tooltip("mean_txt:N", title="mean")])
means = alt.Chart(bd).mark_tick(color=BLUE, thickness=4, size=28, opacity=1).encode(x="mean:Q", y=yenc)
dots = alt.Chart(rd.dropna(subset=["score"])).mark_circle(size=170, color=BLUE, opacity=1, stroke="white", strokeWidth=2).encode(
    x="score:Q", y=yenc, tooltip=[alt.Tooltip("run:N"), alt.Tooltip("score_txt:N", title="score"), alt.Tooltip("cost_txt:N", title="cost")])
crosses = alt.Chart(none).mark_text(text="×", color=RED, fontSize=22, fontWeight=700).encode(
    x="x:Q", y=yenc, tooltip=[alt.Tooltip("run:N"), alt.Tooltip("score_txt:N", title="score")])
twap_rule = alt.Chart(pd.DataFrame({"x": [TWAP], "t": [f"TWAP {lb.fmt(TWAP)}"]}))
rule = twap_rule.mark_rule(color=MUTED, strokeDash=[5, 4], strokeWidth=1.6).encode(x="x:Q")
rule_txt = twap_rule.mark_text(align="center", baseline="bottom", dy=-6, fontSize=12, color=INK2).encode(x="x:Q", y=alt.value(0), text="t:N")
mean_txt = alt.Chart(bd).mark_text(align="left", dx=14, fontSize=14, fontWeight=700, color=INK).encode(x=alt.value(W), y=yenc, text="mean_txt:N")
a_lb = (bars + rule + means + dots + crosses + rule_txt + mean_txt).properties(width=W, height=H)
a_save(a_finish(a_lb, LB_TITLE, LB_SUB, W), "leaderboard")

## 2. Score against the Artificial Analysis Intelligence Index

In [ ]:
sr = runs.dropna(subset=["score"])
r_runs, r_means = sr[["index", "score"]].corr().iloc[0, 1], board[["index", "mean"]].corr().iloc[0, 1]
IX_TITLE = "Capability helps, but it isn't the whole story"
IX_SUB = [f"Final policy on the {SET_LABEL[SCORE_SET]} against the Artificial Analysis Intelligence Index. Dots: runs; dash: model mean.",
          f"Pearson r = {r_runs:.2f} over {len(sr)} runs, {r_means:.2f} over {len(board)} model means."]
nudge = {"gpt-oss-120b": (0, -14), "GLM 5.3": (-12, 0), "DeepSeek V4.1 Flash": (26, 0)}
lab = []
for _, b in board.iterrows():
    sc = runs[runs.model == b.model].score.dropna()
    dx, dy = nudge.get(b.model, (0, 12))
    lab.append({"index": b["index"], "y": sc.min() if dy < 0 else sc.max(), "label": b.model, "dx": dx, "dy": dy})
lab = pd.DataFrame(lab)
y0, y1 = limits(list(runs.score) + [TWAP], pad=0.6)
W, H = 680, 380
xs, ys = alt.Scale(domain=[0, 62], nice=False), alt.Scale(domain=[y0, y1], nice=False, zero=False)
rd = run_tooltip(sr)
pts = alt.Chart(rd).mark_circle(size=150, color=BLUE, opacity=0.85, stroke="white", strokeWidth=2).encode(
    x=alt.X("index:Q", scale=xs, title="Artificial Analysis Intelligence Index"),
    y=alt.Y("score:Q", scale=ys, title=SET_LABEL[SCORE_SET], axis=alt.Axis(labelExpr=VEGA_LABEL, tickCount=7)),
    tooltip=[alt.Tooltip("run:N"), alt.Tooltip("index:Q"), alt.Tooltip("score_txt:N", title="score")])
dash = alt.Chart(board).mark_tick(orient="horizontal", color=INK, thickness=3, size=30, opacity=1).encode(x=alt.X("index:Q", scale=xs), y=alt.Y("mean:Q", scale=ys))
tw = alt.Chart(pd.DataFrame({"y": [TWAP], "t": [f"TWAP baseline {lb.fmt(TWAP)}"], "x": [61.5]}))
a_ix = (tw.mark_rule(color=MUTED, strokeDash=[5, 4], strokeWidth=1.6).encode(y=alt.Y("y:Q", scale=ys))
        + tw.mark_text(align="right", baseline="bottom", dy=-4, fontSize=12, color=INK2).encode(x=alt.X("x:Q", scale=xs), y=alt.Y("y:Q", scale=ys), text="t:N")
        + dash + pts + a_labels(lab, "index", "y", xs, ys)).properties(width=W, height=H)
a_save(a_finish(a_ix, IX_TITLE, IX_SUB, W), "score_vs_index")

## 3. Score against cost

In [ ]:
sc = runs.dropna(subset=["score"])
front = pareto(runs)
best = sc.score.max()
cheap = sc[sc.score >= 0.9 * best].sort_values("cost").iloc[0]
CO_TITLE = f"{SHORT[cheap.model]} gets {cheap.score / best:.0%} of the best score for ${cheap.cost:.2f}"
CO_SUB = [f"Final policy on the {SET_LABEL[SCORE_SET]} against the model cost of the run (Opus at API prices; it ran on a subscription).",
          "Ringed runs and the line: the Pareto frontier, runs no other run beats on both cost and score."]
COST_LABELS = {
    "full": {("MiMo v2.6 Pro", 1): None, ("MiMo v2.6 Pro", 2): (14, 12), ("gpt-oss-120b", 1): (-12, 10),
             ("GPT-6.1 Sol", 1): (-14, 0), ("Opus 5.5", 1): (-11, 0), ("Opus 5.5", 2): (-11, 0)},
    "episodes": {("MiMo v2.6 Pro", 1): (-12, 10), ("MiMo v2.6 Pro", 2): (10, -10), ("gpt-oss-120b", 1): (-12, 10),
                 ("GPT-6.1 Sol", 1): (-14, 0), ("GPT-6.1 Sol", 2): (-14, 0), ("Opus 5.5", 1): (10, -9),
                 ("Opus 5.5", 2): (-11, 6), ("GLM 5.3", 2): (-11, 0)},
}[SCORE_SET]
mimo = sc[sc.model == "MiMo v2.6 Pro"].score
together = mimo.max() - mimo.min() < 0.02 * best
lab = []
for _, r in sc.iterrows():
    off = COST_LABELS.get((r.model, r.rep), (9, 0))
    if off is None:
        continue
    text = "MiMo (both runs)" if r.model == "MiMo v2.6 Pro" and together else SHORT[r.model]
    lab.append({"cost": r.cost, "score": r.score, "label": text, "dx": off[0], "dy": off[1]})
lab = pd.DataFrame(lab)
steps = pd.concat([front[["cost", "score"]], pd.DataFrame({"cost": [20.0], "score": [front.score.iloc[-1]]})], ignore_index=True)
y0, y1 = limits(list(sc.score) + [TWAP], pad=0.6)
W, H = 680, 380
xs = alt.Scale(type="log", domain=[0.05, 20], nice=False)
ys = alt.Scale(domain=[y0, y1], nice=False, zero=False)
rd = run_tooltip(sc)
tw = alt.Chart(pd.DataFrame({"y": [TWAP], "t": [f"TWAP {lb.fmt(TWAP)}"], "x": [19.0]}))
a_co = (tw.mark_rule(color=MUTED, strokeDash=[5, 4], strokeWidth=1.6).encode(y=alt.Y("y:Q", scale=ys))
        + tw.mark_text(align="right", baseline="bottom", dy=-4, fontSize=12, color=INK2).encode(x=alt.X("x:Q", scale=xs), y=alt.Y("y:Q", scale=ys), text="t:N")
        + alt.Chart(steps).mark_line(interpolate="step-after", color=INK, strokeWidth=1.6, opacity=0.55).encode(
            x=alt.X("cost:Q", scale=xs, title="cost of the run (US$, log scale)", axis=alt.Axis(values=[0.1, 1, 10], labelExpr="'$' + datum.value")),
            y=alt.Y("score:Q", scale=ys, title=SET_LABEL[SCORE_SET], axis=alt.Axis(labelExpr=VEGA_LABEL, tickCount=7)))
        + alt.Chart(front).mark_circle(size=520, fillOpacity=0, stroke=INK, strokeWidth=1.3, opacity=0.7).encode(x=alt.X("cost:Q", scale=xs), y=alt.Y("score:Q", scale=ys))
        + alt.Chart(rd).mark_circle(size=150, color=BLUE, opacity=0.9, stroke="white", strokeWidth=2).encode(
            x=alt.X("cost:Q", scale=xs), y=alt.Y("score:Q", scale=ys),
            tooltip=[alt.Tooltip("run:N"), alt.Tooltip("cost_txt:N", title="cost"), alt.Tooltip("score_txt:N", title="score")])
        + a_labels(lab, "cost", "score", xs, ys)).properties(width=W, height=H)
a_save(a_finish(a_co, CO_TITLE, CO_SUB, W), "score_vs_cost")

## 4. Progress over each run

In [ ]:
steps = best_so_far(lb)
subs = lb.subs[lb.subs.valid].assign(episodes=lambda d: d.episodes.clip(lower=0))
run_end = steps.groupby(["model", "rep"]).active_min.max().rename("x").reset_index()   # diamonds sit at each run's end
finals = runs.dropna(subset=["score"]).merge(run_end, on=["model", "rep"])
ep_twap = lb.twap["episodes"]
top = limits(list(subs.episodes) + list(runs.score), pad=0.3, floor=0)[1]
PR_TITLE = "Best submission so far over each two-hour run"
PR_SUB = [f"Lines: best score so far on the 12 feedback episodes, {lb.unit_label} (solid run 1, dashed run 2). Dots: each submission.",
          f"Diamond: final policy on the {SET_LABEL[SCORE_SET]}. Dashed grey: TWAP on the episodes ({lb.fmt(ep_twap)})."]
order = board.model.tolist()
long = pd.concat([steps.assign(kind="step"),
                  subs.rename(columns={"episodes": "best"})[["model", "rep", "active_min", "best"]].assign(kind="sub"),
                  finals[["model", "rep", "x", "score"]].rename(columns={"x": "active_min", "score": "best"}).assign(kind="final"),
                  pd.DataFrame({"model": order, "best": ep_twap, "kind": "twap"})], ignore_index=True)
xs, ys = alt.Scale(domain=[0, 125], nice=False), alt.Scale(domain=[0, top], nice=False)
X = alt.X("active_min:Q", scale=xs, title="active model time (min)", axis=alt.Axis(values=[0, 30, 60, 90, 120]))
Y = alt.Y("best:Q", scale=ys, title="best score so far", axis=alt.Axis(labelExpr=VEGA_LABEL, tickCount=5))
base = alt.Chart(long)
layer = alt.layer(
    base.transform_filter("datum.kind == 'twap'").mark_rule(color=MUTED, strokeDash=[5, 4], strokeWidth=1.3).encode(y=Y),
    base.transform_filter("datum.kind == 'step'").mark_line(color=BLUE, strokeWidth=2.4).encode(
        x=X, y=Y, detail="rep:N", order="order:Q",
        strokeDash=alt.StrokeDash("rep:N", scale=alt.Scale(domain=[1, 2], range=[[1, 0], [6, 4]]), legend=None)),
    base.transform_filter("datum.kind == 'sub'").mark_circle(color=BLUE, size=36, opacity=0.5).encode(
        x=X, y=Y, tooltip=[alt.Tooltip("model:N"), alt.Tooltip("rep:N", title="run"), alt.Tooltip("active_min:Q", format=".0f", title="minutes"),
                           alt.Tooltip("best:Q", format=".1f", title="episode score")]),
    base.transform_filter("datum.kind == 'final'").mark_point(shape="diamond", filled=True, size=130, color=INK, stroke="white", strokeWidth=1.5).encode(
        x=X, y=Y, tooltip=[alt.Tooltip("model:N"), alt.Tooltip("rep:N", title="run"), alt.Tooltip("best:Q", format=".1f", title="final score")]),
).properties(width=250, height=190)
a_pr = layer.facet(facet=alt.Facet("model:N", sort=order, title=None), columns=3, spacing={"row": 28, "column": 22})
a_save(a_finish(a_pr, PR_TITLE, PR_SUB, 800), "progress")

## 5. Feedback episodes against the full holdout

In [ ]:
sc = runs.dropna(subset=["full", "episodes"])
rho = sc[["episodes", "full"]].corr(method="spearman").iloc[0, 1]
EH_TITLE = "The feedback set ranks runs only loosely"
EH_SUB = [f"Each run's final policy on the 12 feedback episodes the agent tuned against, and on the 12-month holdout, {lb.unit_label}.",
          f"Spearman rank correlation {rho:.2f} over {len(sc)} runs. Dashed: equal scores."]
EH_LABELS = {("MiMo v2.6 Pro", 2): (-10, 0), ("MiMo v2.6 Pro", 1): (10, -8), ("Opus 5.5", 1): (10, 8),
             ("gpt-oss-120b", 1): (-10, 0), ("GLM 5.3", 2): (10, -5)}
lab = pd.DataFrame([{"episodes": r.episodes, "full": r.full, "label": f"{SHORT[r.model]} {r.rep}",
                     "dx": EH_LABELS.get((r.model, r.rep), (9, 0))[0], "dy": EH_LABELS.get((r.model, r.rep), (9, 0))[1]}
                    for _, r in sc.iterrows()])
lo, hi = limits(list(sc.full) + list(sc.episodes) + list(lb.twap.values()), pad=1)
sq = pd.DataFrame({"episodes": [lb.twap["episodes"]], "full": [lb.twap["full"]], "t": ["TWAP baseline"]})
S = 460
sc_ = alt.Scale(domain=[lo, hi], nice=False, zero=False)
rd = sc.assign(run=sc.model + " run " + sc.rep.astype(str), e_txt=sc.episodes.map(lb.fmt), f_txt=sc.full.map(lb.fmt))
a_eh = (alt.Chart(pd.DataFrame({"a": [lo, hi]})).mark_line(color=MUTED, strokeDash=[5, 4], strokeWidth=1.4).encode(
            x=alt.X("a:Q", scale=sc_, title="final policy on the 12 feedback episodes", axis=alt.Axis(labelExpr=VEGA_LABEL)),
            y=alt.Y("a:Q", scale=sc_, title="final policy on the 12-month holdout", axis=alt.Axis(labelExpr=VEGA_LABEL)))
        + alt.Chart(sq).mark_square(size=110, color=MUTED).encode(x=alt.X("episodes:Q", scale=sc_), y=alt.Y("full:Q", scale=sc_))
        + alt.Chart(sq).mark_text(dy=-14, fontSize=12, color=INK2).encode(x=alt.X("episodes:Q", scale=sc_), y=alt.Y("full:Q", scale=sc_), text="t:N")
        + alt.Chart(rd).mark_circle(size=150, color=BLUE, opacity=0.9, stroke="white", strokeWidth=2).encode(
            x=alt.X("episodes:Q", scale=sc_), y=alt.Y("full:Q", scale=sc_),
            tooltip=[alt.Tooltip("run:N"), alt.Tooltip("e_txt:N", title="episodes"), alt.Tooltip("f_txt:N", title="12-month holdout")])
        + a_labels(lab, "episodes", "full", sc_, sc_)).properties(width=S, height=S)
a_save(a_finish(a_eh, EH_TITLE, EH_SUB, S), "episodes_vs_holdout")